<a href="https://colab.research.google.com/github/AnkitD-3062/frictionmap/blob/main/frictionmap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**install**



In [1]:
!pip install google-play-scraper pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00


**config**

In [2]:
import pandas as pd
import time
from datetime import datetime, timedelta
from google_play_scraper import reviews, Sort

APPS = {
    "PhonePe": "com.phonepe.app",
    "Paytm": "net.one97.paytm",
    "GPay": "com.google.android.apps.nbu.paisa.user",
    "CRED": "com.dreamplug.androidapp",
    "Groww": "com.nextbillion.groww",
    "Zerodha Kite": "com.zerodha.kite3",
    "YONO SBI": "com.sbi.lotusintouch",
}

CUTOFF = datetime.now() - timedelta(days=395)  # ~13 months
MAX_PER_APP = 6000

Scrape

In [4]:
def scrape_app(name, app_id):
    collected, token = [], None
    while len(collected) < MAX_PER_APP:
        batch, token = reviews(
            app_id,
            lang="en",
            country="in",
            sort=Sort.NEWEST,
            count=200,
            continuation_token=token,
        )
        if not batch:
            break
        collected.extend(batch)
        if batch[-1]["at"] < CUTOFF or token is None:
            break
        time.sleep(1)
    df = pd.DataFrame(collected)
    df["app"] = name
    df = df[df["at"] >= CUTOFF]
    print(f"{name}: {len(df)} reviews")
    return df

frames = []
for name, app_id in APPS.items():
    try:
        frames.append(scrape_app(name, app_id))
    except Exception as e:
        print(f"{name} failed: {e}")

data = pd.concat(frames, ignore_index=True)

PhonePe: 6000 reviews
Paytm: 6000 reviews
GPay: 6000 reviews
CRED: 6000 reviews
Groww: 6000 reviews
Zerodha Kite: 6000 reviews
YONO SBI: 6000 reviews


**Clean and check**

In [5]:
data = data[["app", "content", "score", "at", "reviewCreatedVersion",
             "thumbsUpCount", "replyContent"]].drop_duplicates(subset=["app", "content", "at"])

print(data.shape)
print(data.groupby("app").size())
print(data["score"].value_counts(normalize=True).sort_index())
print("Missing app version:", data["reviewCreatedVersion"].isna().mean().round(2))

(42000, 7)
app
CRED            6000
GPay            6000
Groww           6000
Paytm           6000
PhonePe         6000
YONO SBI        6000
Zerodha Kite    6000
dtype: int64
score
1    0.189262
2    0.027810
3    0.036214
4    0.082548
5    0.664167
Name: proportion, dtype: float64
Missing app version: 0.13


**save**

In [6]:
data.to_csv("fintech_reviews_raw.csv", index=False)

from google.colab import files
files.download("fintech_reviews_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
print(data.shape)

(42000, 7)


**start a clean copy and track row counts**

In [8]:
import re

clean = data.copy()
start_rows = len(clean)
print("Starting rows:", start_rows)

Starting rows: 42000


**drop empty reviews and tidy whitespace**

In [9]:
clean = clean.dropna(subset=["content", "score", "at"])
clean["content"] = clean["content"].astype(str).str.strip()
clean["content"] = clean["content"].str.replace(r"\s+", " ", regex=True)
clean = clean[clean["content"] != ""]
print("After removing empty:", len(clean))

After removing empty: 42000


**remove reviews with no real text (emoji-only, symbols, very short)**

In [10]:
def letter_count(s):
    return len(re.findall(r"[A-Za-z\u0900-\u097F]", s))

def word_count(s):
    return len(re.findall(r"[A-Za-z\u0900-\u097F']+", s))

clean["n_letters"] = clean["content"].apply(letter_count)
clean["n_words"] = clean["content"].apply(word_count)

clean = clean[(clean["n_letters"] >= 5) & (clean["n_words"] >= 3)]
print("After removing too-short:", len(clean))

After removing too-short: 18463


**handle Hindi script and Hinglish**

In [11]:
def devanagari_ratio(s):
    letters = re.findall(r"[A-Za-z\u0900-\u097F]", s)
    if not letters:
        return 0
    return sum("\u0900" <= ch <= "\u097F" for ch in letters) / len(letters)

clean["devanagari_ratio"] = clean["content"].apply(devanagari_ratio)

# Common English words as a rough check for English vs Hinglish
EN_WORDS = set("""the is a an and to of in for it not this that but with my i me you app
payment money bank account login otp kyc update support service worst bad good please
transaction failed fail error problem issue cant cannot can't don't doesn't when why
how very too so are was be have has had will would your no yes""".split())

def english_ratio(s):
    words = re.findall(r"[a-z']+", s.lower())
    if not words:
        return 0
    return sum(w in EN_WORDS for w in words) / len(words)

clean["english_ratio"] = clean["content"].apply(english_ratio)

# Flag, don't delete yet
clean["lang_flag"] = "english"
clean.loc[clean["english_ratio"] < 0.15, "lang_flag"] = "likely_hinglish"
clean.loc[clean["devanagari_ratio"] > 0.3, "lang_flag"] = "hindi_script"

print(clean["lang_flag"].value_counts())
print(clean[clean["lang_flag"] == "likely_hinglish"]["content"].sample(5, random_state=1).tolist())

lang_flag
english            16207
likely_hinglish     2008
hindi_script         248
Name: count, dtype: int64
["without using cred cash segment. it's reflected in credit profile. they haven't followed RBI Norms for loan procedure.", 'screen orientation changes after viewing chart.', 'app ko band kar do baar baar update dete rehte ho isse acha band kar do', 'Bhot acha aap hai. aur simple tarike se use kar shakte hai. me 4 saal se use kar rha hu best aap hai.', 'Regional drama content bhi milta hai']


: decide what to keep

In [12]:
# Always drop Hindi-script reviews (our classifier won't handle them well)
clean = clean[clean["lang_flag"] != "hindi_script"]

# Keep Hinglish for now; filter later with: clean[clean["lang_flag"] == "english"]
print("After language filter:", len(clean))

After language filter: 18215


**Create a normalized text column and fix dates and versions**

In [13]:
def normalize(s):
    s = s.lower()
    s = re.sub(r"(.)\1{3,}", r"\1\1", s)       # sooooo -> soo
    s = re.sub(r"http\S+", "", s)               # remove URLs
    s = re.sub(r"\s+", " ", s).strip()
    return s

clean["text_clean"] = clean["content"].apply(normalize)

clean["at"] = pd.to_datetime(clean["at"])
clean["month"] = clean["at"].dt.to_period("M").astype(str)

clean["reviewCreatedVersion"] = clean["reviewCreatedVersion"].fillna("unknown").astype(str)
clean["has_dev_reply"] = clean["replyContent"].notna()

remove duplicate reviews and check the result

In [14]:
clean = clean.drop_duplicates(subset=["app", "text_clean"])

print(f"Rows: {start_rows} -> {len(clean)} ({len(clean)/start_rows:.0%} kept)")
print(clean.groupby("app").size())
print(clean.groupby("month").size())
print(clean["score"].value_counts(normalize=True).sort_index().round(3))

Rows: 42000 -> 17520 (42% kept)
app
CRED            3275
GPay            1818
Groww           2124
Paytm           1482
PhonePe         1570
YONO SBI        3367
Zerodha Kite    3884
dtype: int64
month
2025-11      44
2025-12     466
2026-01     473
2026-02     504
2026-03     410
2026-04     673
2026-05     918
2026-06     954
2026-07    1574
2026-08    3237
2026-09    8267
dtype: int64
score
1    0.385
2    0.053
3    0.055
4    0.082
5    0.425
Name: proportion, dtype: float64


SAVE

In [15]:
clean = clean.drop(columns=["n_letters", "n_words", "devanagari_ratio", "english_ratio"])
clean.to_csv("fintech_reviews_clean.csv", index=False)

from google.colab import files
files.download("fintech_reviews_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Draw a 200-review sample for hand-labeling**

In [16]:
# Keep this reproducible and spread across apps
sample = (
    clean.groupby("app", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), 30), random_state=42))
)
sample = sample.sample(n=min(len(sample), 200), random_state=42).reset_index(drop=True)

sample["friction_label"] = ""   # you fill this in
sample["notes"] = ""

sample[["app", "score", "content", "friction_label", "notes"]].to_csv("labeling_sample.csv", index=False)
files.download("labeling_sample.csv")
print(sample["app"].value_counts())

/tmp/ipykernel_1611/1339649021.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), 30), random_state=42))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

app
GPay            30
YONO SBI        29
Groww           29
PhonePe         29
Zerodha Kite    28
CRED            28
Paytm           27
Name: count, dtype: int64


In [17]:
print(pd.crosstab(clean["month"], clean["app"]))
print(clean.groupby("app")["at"].agg(["min", "max"]))

app      CRED  GPay  Groww  Paytm  PhonePe  YONO SBI  Zerodha Kite
month                                                             
2025-11     0     0      0      0        0         0            44
2025-12     0     0      0      0        0         0           466
2026-01     0     0      0      0        0         0           473
2026-02     0     0      0      0        0         0           504
2026-03     0     0      0      0        0         0           410
2026-04   299     0      0      0        0         0           374
2026-05   666     0      0      0        0         0           252
2026-06   637     0      0      0        0         0           317
2026-07   577     0      0      0        0       637           360
2026-08   556   306    597      0        0      1452           326
2026-09   540  1512   1527   1482     1570      1278           358
                             min                 max
app                                                 
CRED         2026-04-14

mark the shared window and check sample size

In [18]:
clean["in_cross_window"] = (clean["at"] >= "2026-09-18") & (clean["at"] < "2026-10-01")
print(clean[clean["in_cross_window"]].groupby("app").size())

app
CRED             240
GPay             535
Groww            669
Paytm           1070
PhonePe         1561
YONO SBI         598
Zerodha Kite     152
dtype: int64


In [19]:
# Keep this reproducible and spread across apps
sample = (
    clean.groupby("app", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), 30), random_state=42))
)
sample = sample.sample(n=min(len(sample), 200), random_state=42).reset_index(drop=True)

sample["friction_label"] = ""   # you fill this in
sample["notes"] = ""

sample[["app", "score", "content", "friction_label", "notes"]].to_csv("labeling_sample.csv", index=False)
files.download("labeling_sample.csv")
print(sample["app"].value_counts())

/tmp/ipykernel_1611/1339649021.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), 30), random_state=42))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

app
GPay            30
YONO SBI        29
Groww           29
PhonePe         29
Zerodha Kite    28
CRED            28
Paytm           27
Name: count, dtype: int64


In [20]:
# Extra low-star reviews, excluding ones already in the first sample
first = pd.read_csv("labeling_sample.csv")  # re-upload if the session reset
seen = set(zip(first["app"], first["content"]))

low = clean[(clean["score"] <= 3) & (clean["text_clean"].str.split().str.len() >= 5)]
low = low[[ (a, c) not in seen for a, c in zip(low["app"], low["content"]) ]]

extra = (
    low.groupby("app", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), 22), random_state=7))
    .reset_index(drop=True)
)
extra["friction_label"] = ""
extra["notes"] = ""
extra[["app", "score", "content", "friction_label", "notes"]].to_csv("labeling_sample_extra.csv", index=False)

from google.colab import files
files.download("labeling_sample_extra.csv")
print(len(extra))

/tmp/ipykernel_1611/4264197108.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), 22), random_state=7))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

154


In [21]:
labeled = pd.read_csv("labeling_sample_labeled.csv")
print(labeled.shape)
print(labeled["friction_label"].value_counts())

(200, 5)
friction_label
no_friction        87
performance        24
vague_negative     16
login_access       14
trust_security     10
rewards_offers     10
feature_request     8
payment_failure     8
support             7
other               7
navigation          5
onboarding_kyc      4
Name: count, dtype: int64
